## Island Restriction and Threat Status

Hypothesis Test: Is island restriction associated with whether a bird species is threatened?

H₀: Island restriction and threat status are independent.
H₁: Island restriction and threat status are associated.

We use a chi-square test of independence with α = 0.05.

Threatened species include VU, EN, CR, CR (PE), and CR (PEW).

Non-threatened species include LC and NT. We exclude other or missing conservation statuses, as well as records that dont have a valid island classification.

The test uses species counts. We also compare the threatened percentage within each island group to describe the direction and size of the difference.

The included dataset contains 2,602 island-restricted species and 8,385 species not restricted to islands, so we compare threatened percentages within each group when interpreting and plotting the results to account for such size differences (makes such differences relative)

Because island restriction and threat status are both categorical variables, we use a chi-square test of independence on species counts.



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from IPython.display import display

chi_filename = "BIRDBASE v2025.1 Sekercioglu et al. Final.xlsx"

chi_birds = pd.read_excel(
    chi_filename,
    sheet_name="Data",
    header=1
)

chi_birds = chi_birds[["ISL", "2024 IUCN Red List category"]].copy()

# Parse island classifications and clean conservation labels.
chi_birds["ISL"] = pd.to_numeric(chi_birds["ISL"], errors="coerce")

chi_status = chi_birds["2024 IUCN Red List category"].astype("string")
chi_status = chi_status.str.strip()

chi_status_groups = {
    "LC": "Non-threatened",
    "NT": "Non-threatened",
    "VU": "Threatened",
    "EN": "Threatened",
    "CR": "Threatened",
    "CR (PE)": "Threatened",
    "CR (PEW)": "Threatened"
}

chi_birds["Threat status"] = chi_status.map(chi_status_groups)

# Keep only records with usable classifications.
chi_analysis = chi_birds.dropna(subset=["Threat status"])

chi_valid_island = (
    (chi_analysis["ISL"] == 0) |
    (chi_analysis["ISL"] == 1)
)

chi_analysis = chi_analysis[chi_valid_island].copy()

chi_analysis["Island restriction"] = chi_analysis["ISL"].map({
    0: "Not island-restricted",
    1: "Island-restricted"
})

print("Original species:", len(chi_birds))
print("Included species:", len(chi_analysis))
print("Excluded species:", len(chi_birds) - len(chi_analysis))

display(chi_analysis.head())

FileNotFoundError: [Errno 2] No such file or directory: 'BIRDBASE v2025.1 Sekercioglu et al. Final.xlsx'

In [ ]:
chi_table = pd.crosstab(
    chi_analysis["Island restriction"],
    chi_analysis["Threat status"]
)

print("Observed species counts:")
display(chi_table)

# Both island groups and both threat categories must be present
if chi_table.shape != (2, 2):
    raise ValueError("The test requires both island groups and both threat categories.")

chi_alpha = 0.05

# Pearson chi-square without continuity correction
chi_statistic, chi_p_value, chi_dof, chi_expected = chi2_contingency(
    chi_table,
    correction=False
)

chi_expected_table = pd.DataFrame(
    chi_expected,
    index=chi_table.index,
    columns=chi_table.columns
)

print("Expected counts under independence:")
display(chi_expected_table.round(2))

if chi_expected.min() < 5:
    raise ValueError("An expected count is below 5. Review the test before interpreting it.")

print("Chi-square statistic:", chi_statistic)
print("Degrees of freedom:", chi_dof)
print("P-value:", chi_p_value)

if chi_p_value < chi_alpha:
    print("Reject H₀: evidence of an association between island restriction and threat status.")
else:
    print("Fail to reject H₀: insufficient evidence of an association.")

In [ ]:
chi_group_totals = chi_table.sum(axis=1)

chi_percentages = (
    chi_table["Threatened"] / chi_group_totals
) * 100

chi_summary = pd.DataFrame({
    "Total species": chi_group_totals,
    "Threatened species": chi_table["Threatened"],
    "Threatened (%)": chi_percentages
})

display(chi_summary.round(2))

chi_island_percent = chi_percentages.loc["Island-restricted"]
chi_other_percent = chi_percentages.loc["Not island-restricted"]

chi_difference = chi_island_percent - chi_other_percent

print(
    "Difference (island minus other):",
    round(chi_difference, 2),
    "percentage points"
)

In [ ]:
chi_plot_labels = ["Island-restricted", "Not island-restricted"]
chi_plot_values = [chi_island_percent, chi_other_percent]

plt.figure(figsize=(8, 5))

plt.bar(
    chi_plot_labels,
    chi_plot_values,
    color=["steelblue", "darkorange"],
    width=0.55
)

# Label each bar with its percentage and total number of species.
for chi_i in range(len(chi_plot_labels)):
    chi_group = chi_plot_labels[chi_i]
    chi_percent = chi_plot_values[chi_i]
    chi_total = chi_group_totals.loc[chi_group]

    plt.text(
        chi_i,
        chi_percent + 0.5,
        f"{chi_percent:.2f}%\n(n = {chi_total:,})",
        ha="center"
    )

plt.title("Threatened Bird Species by Island Restriction")
plt.xlabel("Island restriction")
plt.ylabel("Threatened species within group (%)")

plt.ylim(0, max(chi_plot_values) + 8)
plt.tight_layout()
plt.show()

## Conclusion

Among 10,987 included species, 23.02% of island-restricted species were threatened, compared with 8.49% of species not restricted to islands. The difference was 14.53 percentage points. This also means that the proportion of threatened species among island-restricted birds is about 2.7 times that among birds not restricted to islands.

All expected counts exceeded 5, supporting the use of the chi-square approximation. Hooray!

The chi-square test produced chi sqr statistic of 398.92, with 1 degree of freedom and p ≈ 9.46 × 10⁻⁸⁹. At α = 0.05, we reject the null hypothesis. This provides statistical evidence of an association between island restriction and threat status, with island-restricted species having a higher threatened proportion in the analyzed dataset.

This does not establish causation. Its results apply to the included species and depend on our conservation-status grouping and exclusions. Related species share evolutionary history, so the assumption of independent observations is not perfect.